# 🎓 UPPA Smart Assistant — Notebook 04
## **Évaluer et améliorer le retrieval : BM25, recherche hybride, reranking**

**Mini-projet M2** · Master Industry 4.0 · Master Systèmes informatiques pour le génie de la logistique industrielle et des services

**Domaines documentaires :** 🏫 Formations · 📝 Admissions · 🔬 Recherche

> **Objectif :** comparer, sur **les mêmes questions annotées**, quatre façons de retrouver des documents — dense, lexicale (BM25), hybride (RRF) et hybride + reranking — puis décider laquelle retenir **à partir de mesures**, pas d'impressions.
>
> Projet pédagogique indépendant de l'Université de Pau et des Pays de l'Adour (UPPA). Les données **fictives** ne constituent pas des informations institutionnelles.

**Entrée :** l'archive d'index du Notebook 02 (`UPPA_Notebook02_index_*.zip`) ou de l'Atelier 02+ (`UPPA_Notebook02_V3_index_*.zip`), et un fichier de **questions annotées par vous**.
**Sortie :** tableau comparatif, analyse d'erreurs, grille de 45 questions à compléter et archive d'évaluation pour le rapport.

### Compétences visées

1. Construire une recherche **lexicale BM25** adaptée au français, à côté de la recherche dense.
2. Fusionner deux classements avec **Reciprocal Rank Fusion (RRF)** sans additionner des scores incomparables.
3. Appliquer un **reranking par cross-encoder** sur un petit nombre de candidats.
4. Calculer **Precision@k, Recall@k, Hit@k et MRR** au niveau des documents sources.
5. Lire un tableau comparatif avec prudence : gains, **régressions** et taille de l'échantillon.
6. Séparer l'évaluation du **retrieval** de celle de la **génération** (fidélité, citations, abstention).

**Pré-requis :** Notebooks 01 à 03. Ce notebook ne collecte aucune page et n'appelle aucun LLM.

**Règle de l'atelier :** sur le corpus réel, **aucun score n'est affiché tant que les références n'ont pas été annotées à la main**. Annoter en regardant les résultats du moteur fausserait la mesure.

---
## Étape 0 — Installation

`rank-bm25` fournit BM25 ; `sentence-transformers` fournit l'encodeur du Notebook 02 et le cross-encoder. Aucune clé API n'est nécessaire ; les modèles sont téléchargés à la première exécution.

In [ ]:
%pip -q install "sentence-transformers>=3,<6" "rank-bm25>=0.2.2,<0.3" "pandas>=2" "matplotlib>=3.7" "numpy>=1.26"

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
from urllib.parse import urlparse
from IPython.display import display
import json, re, zipfile, time, unicodedata, io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

BASE = Path('/content/UPPA_RAG_NB04' if Path('/content').exists() else './UPPA_RAG_NB04').resolve()
EXPORT_DIR = BASE/'exports'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
DOMAINS = ('formation','admission','recherche')
OFFICIAL_HOSTS = {'formation.univ-pau.fr','recherche.univ-pau.fr'}
print('Dossier de travail :', BASE)

---
## Étape 1 — Charger l'index et les questions annotées

- **`INPUT_ZIP`** : archive du Notebook 02 (V2) ou de l'Atelier 02+ (V3). Seuls le manifeste et les chunks sont lus ; les embeddings sont recalculés avec le modèle indiqué par le manifeste.
- **`ANNOTATIONS_CSV`** : votre fichier de questions annotées (colonnes `question` et `reference_document_ids`, identifiants séparés par `;`). Les grilles produites par les Notebooks 02 et 02 V3 conviennent.
- **`NB03_ZIP`** *(facultatif)* : archive du Notebook 03 dont vous avez **complété à la main** la grille de fidélité.

Sans archive, le notebook utilise 13 chunks **explicitement fictifs** pour dérouler la méthode.

In [ ]:
# ------------ PARAMÈTRES DE L'ÉTUDIANT ------------
INPUT_ZIP = ''          # ex. '/content/UPPA_Notebook02_V3_index_uppa.zip'
ANNOTATIONS_CSV = ''    # ex. '/content/questions_a_annoter_uppa.csv' (complété à la main)
NB03_ZIP = ''           # ex. '/content/UPPA_Notebook03_reponses_uppa.zip' (grille de fidélité complétée)
UPLOAD_WITH_COLAB = False
K = 5                   # nombre de DOCUMENTS distincts évalués par question
N_CANDIDATES = 30       # nombre de chunks candidats par méthode avant fusion / reranking
RRF_K = 60              # constante de lissage de RRF
USE_CROSS_ENCODER = True
MIN_ANNOTATED = 5       # en dessous, aucun score n'est affiché sur le corpus réel

if UPLOAD_WITH_COLAB:
    from google.colab import files
    for name in files.upload():   # sélectionner l'archive d'index, le CSV annoté et, si besoin, l'archive NB03
        path = str(Path.cwd()/name)
        if name.endswith('.csv'): ANNOTATIONS_CSV = path
        elif 'Notebook03' in name: NB03_ZIP = path
        elif name.endswith('.zip'): INPUT_ZIP = path
    print('Index :', INPUT_ZIP, '| annotations :', ANNOTATIONS_CSV, '| NB03 :', NB03_ZIP)

def find_first(patterns):
    hits = []
    for root in (Path('/content'), Path.cwd()):
        if root.is_dir():
            for pattern in patterns: hits.extend(root.glob(pattern))
    # Priorité : corpus réel avant fictif, index V3 avant V2.
    hits.sort(key=lambda p: ('synthetic_demo' in p.name, '_V3_' not in p.name, p.name))
    return str(hits[0]) if hits else ''

if not INPUT_ZIP:
    INPUT_ZIP = find_first(['UPPA_Notebook02_V3_index_*.zip', 'UPPA_Notebook02_index_*.zip'])
if not NB03_ZIP:
    NB03_ZIP = find_first(['UPPA_Notebook03_reponses_*.zip'])
print('Index utilisé :', INPUT_ZIP or '(aucun : démonstration fictive intégrée)')

In [ ]:
# DÉMONSTRATION EXPLICITEMENT FICTIVE : les 13 chunks du Notebook 03, aucune information réelle sur l'UPPA.
DEMO_CHUNKS = json.loads(r'''[
 {
  "chunk_id": "demo_ffe9847450c745d3_0001_1b83f2370476c68a",
  "document_id": "demo_ffe9847450c745d3",
  "domain": "formation",
  "title": "Master informatique — exemple fictif",
  "source_url": null,
  "collected_at": "2026-11-09T23:05:47+00:00",
  "academic_years_detected": [],
  "dataset_kind": "synthetic_demo",
  "section": "Master informatique — exemple fictif / Présentation",
  "chunk_type": "prose",
  "text": "Formation expérimentale qui illustre les systèmes d'information, l'IA et l'informatique industrielle.",
  "char_count": 101
 },
 {
  "chunk_id": "demo_ffe9847450c745d3_0002_73e5fb8e0f8328f0",
  "document_id": "demo_ffe9847450c745d3",
  "domain": "formation",
  "title": "Master informatique — exemple fictif",
  "source_url": null,
  "collected_at": "2026-11-09T23:05:47+00:00",
  "academic_years_detected": [],
  "dataset_kind": "synthetic_demo",
  "section": "Master informatique — exemple fictif / Parcours",
  "chunk_type": "prose",
  "text": "* Industry 4.0 — cas fictif\n* Logistique et services — cas fictif",
  "char_count": 65
 },
 {
  "chunk_id": "demo_ffe9847450c745d3_0003_b9ebc76ed50c5c5d",
  "document_id": "demo_ffe9847450c745d3",
  "domain": "formation",
  "title": "Master informatique — exemple fictif",
  "source_url": null,
  "collected_at": "2026-11-09T23:05:47+00:00",
  "academic_years_detected": [],
  "dataset_kind": "synthetic_demo",
  "section": "Master informatique — exemple fictif / Compétences",
  "chunk_type": "prose",
  "text": "Concevoir des applications et analyser des données.",
  "char_count": 51
 },
 {
  "chunk_id": "demo_494779650bc67814_0001_85fd4539794ba687",
  "document_id": "demo_494779650bc67814",
  "domain": "formation",
  "title": "Parcours Industry 4.0 — exemple fictif",
  "source_url": null,
  "collected_at": "2026-11-09T23:05:47+00:00",
  "academic_years_detected": [],
  "dataset_kind": "synthetic_demo",
  "section": "Parcours Industry 4.0 — exemple fictif / Objectifs",
  "chunk_type": "prose",
  "text": "Explorer la maintenance prédictive, les capteurs et le pilotage de production.",
  "char_count": 78
 },
 {
  "chunk_id": "demo_494779650bc67814_0002_31faa4bd472143c0",
  "document_id": "demo_494779650bc67814",
  "domain": "formation",
  "title": "Parcours Industry 4.0 — exemple fictif",
  "source_url": null,
  "collected_at": "2026-11-09T23:05:47+00:00",
  "academic_years_detected": [],
  "dataset_kind": "synthetic_demo",
  "section": "Parcours Industry 4.0 — exemple fictif / Tableau des unités",
  "chunk_type": "table",
  "text": "| UE | Thème |\n| --- | --- |\n| UE1 | Capteurs |\n| UE2 | Données industrielles |",
  "char_count": 79
 },
 {
  "chunk_id": "demo_c593913670f65f7d_0001_55f12d6e6c012f38",
  "document_id": "demo_c593913670f65f7d",
  "domain": "admission",
  "title": "Admission en master — exemple fictif",
  "source_url": null,
  "collected_at": "2026-11-09T23:05:47+00:00",
  "academic_years_detected": [],
  "dataset_kind": "synthetic_demo",
  "section": "Admission en master — exemple fictif / Documents",
  "chunk_type": "prose",
  "text": "* Curriculum vitae\n* Relevés de notes",
  "char_count": 37
 },
 {
  "chunk_id": "demo_c593913670f65f7d_0002_b30be8469d62d794",
  "document_id": "demo_c593913670f65f7d",
  "domain": "admission",
  "title": "Admission en master — exemple fictif",
  "source_url": null,
  "collected_at": "2026-11-09T23:05:47+00:00",
  "academic_years_detected": [],
  "dataset_kind": "synthetic_demo",
  "section": "Admission en master — exemple fictif / Procédure",
  "chunk_type": "prose",
  "text": "Consulter les modalités officielles, différentes selon le profil et l'année.\n\nLes dates citées dans ce document sont fictives.",
  "char_count": 126
 },
 {
  "chunk_id": "demo_d1e2dbcf93b1520d_0001_5380e4f5bfb2206a",
  "document_id": "demo_d1e2dbcf93b1520d",
  "domain": "admission",
  "title": "Calendrier d'admission — exemple fictif",
  "source_url": null,
  "collected_at": "2026-11-09T23:05:47+00:00",
  "academic_years_detected": [],
  "dataset_kind": "synthetic_demo",
  "section": "Calendrier d'admission — exemple fictif / Dates illustratives",
  "chunk_type": "table",
  "text": "| Phase | Date fictive |\n| --- | --- |\n| Candidature | 15 janvier 2030 |\n| Réponse | 20 mars 2030 |",
  "char_count": 99
 },
 {
  "chunk_id": "demo_d1e2dbcf93b1520d_0002_431815094ecba775",
  "document_id": "demo_d1e2dbcf93b1520d",
  "domain": "admission",
  "title": "Calendrier d'admission — exemple fictif",
  "source_url": null,
  "collected_at": "2026-11-09T23:05:47+00:00",
  "academic_years_detected": [],
  "dataset_kind": "synthetic_demo",
  "section": "Calendrier d'admission — exemple fictif / Dates illustratives",
  "chunk_type": "prose",
  "text": "Ces dates ne concernent aucune campagne officielle.",
  "char_count": 51
 },
 {
  "chunk_id": "demo_432ebb2fd9f03e32_0001_37530158c68bb093",
  "document_id": "demo_432ebb2fd9f03e32",
  "domain": "recherche",
  "title": "Laboratoire Systèmes numériques — exemple fictif",
  "source_url": null,
  "collected_at": "2026-11-09T23:05:47+00:00",
  "academic_years_detected": [],
  "dataset_kind": "synthetic_demo",
  "section": "Laboratoire Systèmes numériques — exemple fictif / Axes de recherche",
  "chunk_type": "prose",
  "text": "Réseaux, intelligence artificielle, simulation et systèmes cyber-physiques.",
  "char_count": 75
 },
 {
  "chunk_id": "demo_432ebb2fd9f03e32_0002_71032d7eafe5573b",
  "document_id": "demo_432ebb2fd9f03e32",
  "domain": "recherche",
  "title": "Laboratoire Systèmes numériques — exemple fictif",
  "source_url": null,
  "collected_at": "2026-11-09T23:05:47+00:00",
  "academic_years_detected": [],
  "dataset_kind": "synthetic_demo",
  "section": "Laboratoire Systèmes numériques — exemple fictif / Équipes",
  "chunk_type": "prose",
  "text": "Equipe Méthodes et Données.",
  "char_count": 27
 },
 {
  "chunk_id": "demo_f52ce7bf39c1439e_0001_fa13f4279274d706",
  "document_id": "demo_f52ce7bf39c1439e",
  "domain": "recherche",
  "title": "Projet Logistique intelligente — exemple fictif",
  "source_url": null,
  "collected_at": "2026-11-09T23:05:47+00:00",
  "academic_years_detected": [],
  "dataset_kind": "synthetic_demo",
  "section": "Projet Logistique intelligente — exemple fictif / Thématiques",
  "chunk_type": "prose",
  "text": "* Traçabilité\n* Optimisation d'entrepôt\n* Recommandation de services",
  "char_count": 68
 },
 {
  "chunk_id": "demo_f52ce7bf39c1439e_0002_641c96212296579a",
  "document_id": "demo_f52ce7bf39c1439e",
  "domain": "recherche",
  "title": "Projet Logistique intelligente — exemple fictif",
  "source_url": null,
  "collected_at": "2026-11-09T23:05:47+00:00",
  "academic_years_detected": [],
  "dataset_kind": "synthetic_demo",
  "section": "Projet Logistique intelligente — exemple fictif / Partenariats",
  "chunk_type": "prose",
  "text": "Exemple pédagogique sans partenaires réels.",
  "char_count": 43
 }
]''')
print('Chunks de démonstration fictive :', len(DEMO_CHUNKS))

In [ ]:
MAX_MEMBER_BYTES = 80_000_000

def read_member(zip_path, pattern):
    """Lit UN fichier de l'archive, sans rien extraire sur le disque."""
    with zipfile.ZipFile(zip_path) as z:
        matches = [i for i in z.infolist() if re.search(pattern, i.filename.replace('\\','/')) and not i.is_dir()]
        if len(matches) != 1: raise ValueError(f'Fichier attendu introuvable ou ambigu : {pattern}')
        if matches[0].file_size > MAX_MEMBER_BYTES: raise ValueError('Fichier interne trop volumineux.')
        return z.read(matches[0])

def load_index(zip_path):
    if zip_path:
        manifest = json.loads(read_member(zip_path, r'(?:^|/)exports/index_manifest\.json$').decode('utf-8'))
        if manifest.get('schema_version') != 'uppa_rag_index_v1':
            raise ValueError('Manifeste inattendu : fournir une archive du Notebook 02 ou 02 V3.')
        raw = read_member(zip_path, r'(?:^|/)exports/chunks_(?:uppa|synthetic_demo)\.jsonl$')
        chunks = [json.loads(line) for line in raw.decode('utf-8-sig').splitlines() if line.strip()]
    else:
        manifest = {'schema_version':'uppa_rag_index_v1', 'dataset_kind':'synthetic_demo',
                    'embedding_model':'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
                    'embedding_dimension':384, 'note':'Démo fictive intégrée au Notebook 04'}
        chunks = list(DEMO_CHUNKS)
    kind = manifest.get('dataset_kind')
    if kind not in ('live','synthetic_demo'): raise ValueError('Type de corpus inattendu.')
    required = {'chunk_id','document_id','domain','title','section','text','dataset_kind','source_url'}
    if not chunks or len(chunks) > 150_000: raise ValueError('Nombre de chunks invalide.')
    for c in chunks:
        if not required.issubset(c): raise ValueError('Schéma de chunk incorrect.')
        if c['domain'] not in DOMAINS or not str(c['text']).strip(): raise ValueError('Chunk vide ou domaine invalide.')
        if c['dataset_kind'] != kind: raise ValueError('Mélange de documents fictifs et réels interdit.')
        if kind == 'live':
            u = urlparse(str(c.get('source_url') or ''))
            if u.scheme != 'https' or u.hostname not in OFFICIAL_HOSTS:
                raise ValueError('Un document réel doit conserver une source UPPA valide.')
    if len({c['chunk_id'] for c in chunks}) != len(chunks): raise ValueError('Identifiants de chunks dupliqués.')
    return chunks, manifest

CHUNKS, MANIFEST = load_index(INPUT_ZIP)
DATASET_KIND = MANIFEST['dataset_kind']
SUFFIX = 'uppa' if DATASET_KIND == 'live' else 'synthetic_demo'
EMBEDDING_MODEL = MANIFEST['embedding_model']
DOCS = {}
for c in CHUNKS:
    DOCS.setdefault(c['document_id'], {'document_id':c['document_id'], 'domain':c['domain'],
                                        'title':c.get('full_title') or c['title'], 'source_url':c.get('source_url')})
print('Corpus :', DATASET_KIND, '| chunks :', len(CHUNKS), '| documents :', len(DOCS))
print('Chunks par domaine :', dict(Counter(c['domain'] for c in CHUNKS)))
print('Version du découpage :', MANIFEST.get('experiment_version', 'NB02 (V2)'))
if DATASET_KIND != 'live':
    print('⚠️ Corpus FICTIF : aucun résultat de ce notebook ne décrit le site de l’UPPA.')

---
## Étape 2 — Stratégie A : la recherche dense (référence)

C'est la recherche des Notebooks 02 et 03 : la question et les chunks sont encodés par **le même modèle**, et les vecteurs normalisés sont comparés par produit scalaire (= similarité cosinus).

Ici, les vecteurs tiennent dans une matrice `numpy` : sur quelques milliers de chunks, le calcul exact est immédiat et donne le même classement que Qdrant en mode local. Qdrant reste le bon outil pour l'assistant ; la matrice suffit pour **évaluer**.

📌 Le ré-encodage de quelques milliers de chunks prend une à quelques minutes sur CPU.

In [ ]:
encoder = SentenceTransformer(EMBEDDING_MODEL)

def text_for_embedding(chunk):
    # Strictement identique aux Notebooks 02 et 03.
    return f"Titre : {chunk['title']}\nSection : {chunk['section']}\n{chunk['text']}"

t0 = time.perf_counter()
MATRIX = np.asarray(encoder.encode([text_for_embedding(c) for c in CHUNKS], batch_size=32, show_progress_bar=True,
                                   convert_to_numpy=True, normalize_embeddings=True), dtype=np.float32)
assert MATRIX.shape[0] == len(CHUNKS) and np.all(np.isfinite(MATRIX))
if MANIFEST.get('embedding_dimension'):
    assert MATRIX.shape[1] == int(MANIFEST['embedding_dimension']), 'Dimension différente de celle du Notebook 02.'
DOMAIN_OF = np.array([c['domain'] for c in CHUNKS])
print('Matrice :', MATRIX.shape, '| encodage :', round(time.perf_counter()-t0, 1), 's')

def top_indices(scores, domain=None, n=N_CANDIDATES, minimum=None):
    """Indices des n meilleurs chunks (ordre décroissant), éventuellement restreints à un domaine."""
    scores = np.asarray(scores, dtype=float).copy()
    if domain: scores[DOMAIN_OF != domain] = -np.inf
    order = np.argsort(-scores, kind='stable')[:n]
    return [int(i) for i in order if np.isfinite(scores[i]) and (minimum is None or scores[i] > minimum)]

def dense_scores(question):
    q = np.asarray(encoder.encode(question, normalize_embeddings=True), dtype=np.float32)
    return MATRIX @ q

def dense_ranking(question, domain=None, n=N_CANDIDATES):
    return top_indices(dense_scores(question), domain, n)

def show_ranking(indices, n=5):
    display(pd.DataFrame([{'rang':r, 'domaine':CHUNKS[i]['domain'], 'titre':CHUNKS[i]['title'][:60],
                           'section':CHUNKS[i]['section'][-50:], 'extrait':CHUNKS[i]['text'][:110].replace('\n',' ')}
                          for r, i in enumerate(indices[:n], 1)]))

EXEMPLE = 'Quelles formations abordent les capteurs et la maintenance industrielle ?'
print('Question :', EXEMPLE)
show_ranking(dense_ranking(EXEMPLE))

---
## Étape 3 — Stratégie B : la recherche lexicale BM25

BM25 note un passage d'après les **mots de la question qu'il contient** : un terme rare dans le corpus (un code d'UE, un sigle comme `SIGLIS`, un nom de laboratoire) pèse plus qu'un mot fréquent, et un passage long n'est pas avantagé indûment.

- **Force :** les termes exacts, sigles, codes et noms propres, que la recherche dense « lisse » parfois.
- **Faiblesse :** aucune notion de synonymie — « candidater » ne retrouve pas « dépôt de dossier ».

BM25 dépend entièrement de la **tokenisation**. Pour le français, nous mettons en minuscules, retirons les accents, écartons quelques mots vides et supprimons le `s`/`x` final des mots longs (pluriels). C'est volontairement rudimentaire : observez ses erreurs.

In [ ]:
STOPWORDS = set("""a au aux avec ce ces cette dans de des du elle en et est il ils je la le les leur leurs lui ma mais me
mes mon ne nos notre nous on ou où par pas pour qu que quel quelle quelles quels qui sa se ses son sont sur ta te tes
ton tu un une vos votre vous y d l s n c j m t comment quoi dont ainsi etre avoir""".split())
TOKEN_RE = re.compile(r"[a-z0-9]+(?:[-.][a-z0-9]+)*")

def tokenize(text):
    text = unicodedata.normalize('NFKD', text.casefold())
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    tokens = []
    for tok in TOKEN_RE.findall(text):
        if tok in STOPWORDS or len(tok) < 2: continue
        if len(tok) > 4 and tok[-1] in 'sx' and not any(ch.isdigit() for ch in tok): tok = tok[:-1]   # pluriel simple
        tokens.append(tok)
    return tokens

assert tokenize('Les systèmes industriels du parcours SIGLIS (UE-12)') == ['systeme','industriel','parcour','sigli','ue-12']

def text_for_bm25(chunk):
    return f"{chunk['title']} {chunk['section']} {chunk['text']}"

bm25 = BM25Okapi([tokenize(text_for_bm25(c)) or ['_vide_'] for c in CHUNKS])

def bm25_ranking(question, domain=None, n=N_CANDIDATES):
    tokens = tokenize(question)
    if not tokens: return []
    # Un score nul signifie « aucun mot commun » : ces chunks ne sont pas des candidats.
    return top_indices(bm25.get_scores(tokens), domain, n, minimum=0.0)

print('Tokens de la question :', tokenize(EXEMPLE))
show_ranking(bm25_ranking(EXEMPLE))

### À vous de jouer

1. Trouvez une question où BM25 fait **mieux** que la recherche dense (indice : un sigle, un code, un nom propre).
2. Trouvez une question où BM25 **échoue** parce que la page n'emploie pas les mots de la question.
3. La suppression du `s` final transforme « parcours » en « parcour » et « SIGLIS » en « sigli ». Est-ce gênant ? Pourquoi la même transformation doit-elle être appliquée à la question et aux documents ?

---
## Étape 4 — Stratégie C : fusion hybride par RRF

Un score BM25 (non borné) et un cosinus (entre −1 et 1) ne sont **pas comparables** : les additionner n'a pas de sens. **Reciprocal Rank Fusion** n'utilise que les **rangs** :

\[
\operatorname{RRF}(d)=\sum_{\text{classement } r}\frac{1}{k+\operatorname{rang}_r(d)}
\]

Un chunk bien classé par les deux méthodes passe devant un chunk excellent pour une seule. La constante `k` (60 par défaut) atténue l'avantage des tout premiers rangs.

In [ ]:
def rrf(rankings, k=RRF_K):
    scores = defaultdict(float)
    for ranking in rankings:
        for rank, item in enumerate(ranking, 1):
            scores[item] += 1.0/(k+rank)
    return sorted(scores, key=lambda item: (-scores[item], item))

# Vérifications du calcul sur des listes jouets
assert rrf([['a','b','c'], ['b','a','d']])[:2] == ['a','b']        # ex æquo départagés par l'identifiant
assert rrf([['a','b'], ['c','b']])[0] == 'b'                         # présent dans les deux listes
assert rrf([['a','b']]) == ['a','b'] and rrf([[], []]) == []

def hybrid_ranking(question, domain=None, n=N_CANDIDATES):
    return rrf([dense_ranking(question, domain, n), bm25_ranking(question, domain, n)])[:n]

show_ranking(hybrid_ranking(EXEMPLE))

---
## Étape 5 — Stratégie D : reranking par cross-encoder

Un **cross-encoder** lit la paire *(question, passage)* en une seule passe et produit un score de pertinence. Il est bien plus précis qu'une comparaison de deux vecteurs calculés séparément, mais aussi bien plus lent : on ne l'applique qu'aux **candidats de l'hybride**.

Modèle retenu : `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1` (multilingue, le même qu'à l'Atelier 3B).

**Limite structurelle :** un document absent des candidats ne sera jamais retrouvé par le reranker. Le rappel des candidats borne le résultat final.

**Ressources limitées ?** Mettez `USE_CROSS_ENCODER = False` : la stratégie D sera marquée *non exécutée*, sans mesure de remplacement.

In [ ]:
reranker = None
RERANK_MODEL = 'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1'
if USE_CROSS_ENCODER:
    from sentence_transformers import CrossEncoder
    try:
        reranker = CrossEncoder(RERANK_MODEL)
        print('Cross-encoder chargé :', RERANK_MODEL)
    except Exception as exc:
        print('Reranking indisponible :', repr(exc))
        print('La stratégie D sera ignorée ; aucun score de reranking ne sera inventé.')

def rerank_ranking(question, domain=None, n=N_CANDIDATES):
    if reranker is None: raise RuntimeError('Cross-encoder non chargé.')
    candidates = hybrid_ranking(question, domain, n)
    if not candidates: return []
    pairs = [(question, text_for_bm25(CHUNKS[i])) for i in candidates]
    scores = np.asarray(reranker.predict(pairs, batch_size=16), dtype=float).reshape(-1)
    order = sorted(range(len(candidates)), key=lambda j: (-float(scores[j]), candidates[j]))
    return [candidates[j] for j in order]

if reranker is not None:
    show_ranking(rerank_ranking(EXEMPLE))

---
## Étape 6 — Passer des chunks aux documents

Vos annotations désignent des **documents** (`document_id`), alors que les moteurs classent des **chunks**. Pour comparer équitablement :

- chaque stratégie produit jusqu'à `N_CANDIDATES` chunks ;
- on conserve la **première apparition de chaque document** ;
- on évalue les `K` premiers documents distincts.

Six chunks de la même fiche ne valent donc pas six documents. Ce choix doit figurer dans votre rapport.

In [ ]:
def to_documents(chunk_ranking, k=None):
    seen = []
    for i in chunk_ranking:
        doc_id = CHUNKS[i]['document_id']
        if doc_id not in seen: seen.append(doc_id)
        if k and len(seen) >= k: break
    return seen

STRATEGIES = {
    'A — Dense':            dense_ranking,
    'B — BM25':             bm25_ranking,
    'C — Hybride RRF':      hybrid_ranking,
}
if reranker is not None:
    STRATEGIES['D — Hybride + reranking'] = rerank_ranking

apercu = []
for name, fn in STRATEGIES.items():
    docs = to_documents(fn(EXEMPLE), K)
    apercu.append({'stratégie':name, **{f'doc {r}':DOCS[d]['title'][:38] for r, d in enumerate(docs[:3], 1)}})
print('Question :', EXEMPLE)
display(pd.DataFrame(apercu))

---
## Étape 7 — Les questions annotées

Le mini-projet demande **45 questions** : 12 par domaine et 9 transversales, de types variés (factuelle, comparative, multi-sources, temporelle, **sans réponse dans le corpus**).

**Protocole :**
1. écrire la question **avant** d'interroger le moteur ;
2. lire les pages du corpus et noter dans `reference_document_ids` les documents qui permettent de répondre (séparés par `;`) — le catalogue `catalogue_documents_*.csv` donne les identifiants ;
3. pour une question sans réponse dans le corpus, laisser les références vides et mettre `type = sans_reponse` ;
4. passer `statut_annotation` à `annote`.

La cellule suivante écrit une **grille de 45 lignes à compléter** et le catalogue des documents. Sur le corpus fictif, cinq questions de démonstration sont annotées pour montrer le déroulement.

In [ ]:
STARTERS = [
 ('formation','factuelle','Quels parcours sont proposés dans le Master Informatique ?'),
 ('formation','factuelle','Quels objectifs et compétences présente le parcours Industry 4.0 ?'),
 ('formation','comparative','En quoi le parcours Industry 4.0 et le parcours logistique diffèrent-ils ?'),
 ('admission','factuelle','Quelle procédure de candidature est indiquée pour le master 1 ?'),
 ('admission','temporelle','Quelles dates de candidature sont précisées, et pour quelle année universitaire ?'),
 ('admission','sans_reponse','Quel est le taux d’acceptation en master 2 cette année ?'),
 ('recherche','factuelle','Quels laboratoires travaillent sur l’informatique et les systèmes intelligents ?'),
 ('recherche','multi-sources','Quelles thématiques scientifiques sont communes à plusieurs laboratoires ?'),
 ('transversal','multi-sources','Quelles formations abordent les systèmes industriels, comment candidater et quels laboratoires sont proches de ce thème ?'),
]
QUOTAS = {'formation':12, 'admission':12, 'recherche':12, 'transversal':9}
rows = []
for domain, quota in QUOTAS.items():
    starters = [s for s in STARTERS if s[0] == domain]
    for n in range(quota):
        kind, question = (starters[n][1], starters[n][2]) if n < len(starters) else ('', '')
        rows.append({'question_id':f'{domain[:3].upper()}{n+1:02d}', 'domain_cible':domain, 'type':kind, 'question':question,
                     'reference_document_ids':'', 'reponse_reference':'', 'justification':'', 'statut_annotation':'a_annoter'})
grid_path = EXPORT_DIR/f'questions_45_a_annoter_{SUFFIX}.csv'
pd.DataFrame(rows).to_csv(grid_path, index=False, encoding='utf-8-sig')
catalog_path = EXPORT_DIR/f'catalogue_documents_{SUFFIX}.csv'
pd.DataFrame(DOCS.values()).to_csv(catalog_path, index=False, encoding='utf-8-sig')
print('Grille de 45 questions à compléter :', grid_path)
print('Catalogue des identifiants de documents :', catalog_path)

def parse_refs(value):
    if not isinstance(value, str): return set()
    return {s.strip() for s in value.split(';') if s.strip()}

if ANNOTATIONS_CSV:
    labels = pd.read_csv(ANNOTATIONS_CSV, dtype=str).fillna('')
    origin = 'annotations fournies'
elif DATASET_KIND == 'synthetic_demo':
    def demo_ids(part):
        return ';'.join(d for d, meta in DOCS.items() if part.casefold() in meta['title'].casefold())
    demo = [('formation','factuelle','Quels enseignements portent sur les capteurs ?','Industry 4.0'),
            ('admission','factuelle','Quelles pièces joindre à un dossier de master ?','Admission en master'),
            ('admission','temporelle','Où trouver les échéances de dépôt des candidatures ?','Calendrier'),
            ('recherche','factuelle','Quelle équipe étudie les réseaux et la simulation ?','Laboratoire Systèmes'),
            ('recherche','factuelle','Quel projet traite de la traçabilité en entrepôt ?','Logistique intelligente'),
            ('admission','sans_reponse','Quel est le taux d’acceptation en master 2 ?','__aucun__')]
    labels = pd.DataFrame([{'question_id':f'D{i:02d}', 'domain_cible':d, 'type':t, 'question':q,
                            'reference_document_ids':demo_ids(part), 'statut_annotation':'demo_fictive'}
                           for i, (d, t, q, part) in enumerate(demo, 1)])
    origin = 'démonstration FICTIVE'
else:
    labels = pd.DataFrame(columns=['question_id','domain_cible','type','question','reference_document_ids'])
    origin = 'aucune annotation fournie'

for column in ('question_id','domain_cible','type'):
    if column not in labels.columns: labels[column] = ''
EVAL_CASES, NO_ANSWER_CASES = [], []
for n, row in enumerate(labels.to_dict('records'), 1):
    question = str(row.get('question','')).strip()
    if not question: continue
    refs = parse_refs(row.get('reference_document_ids',''))
    unknown = refs - set(DOCS)
    if unknown: raise ValueError(f'Identifiants de référence absents du corpus : {sorted(unknown)}')
    case = {'question_id':row.get('question_id') or f'Q{n:02d}', 'domain_cible':row.get('domain_cible') or 'non précisé',
            'type':row.get('type') or 'non précisé', 'question':question, 'refs':refs}
    if refs: EVAL_CASES.append(case)
    elif str(row.get('type','')).strip().casefold() == 'sans_reponse': NO_ANSWER_CASES.append(case)

CAN_SCORE = bool(EVAL_CASES) and (DATASET_KIND != 'live' or len(EVAL_CASES) >= MIN_ANNOTATED)
print(f'Origine : {origin} | questions avec références : {len(EVAL_CASES)} | questions sans réponse : {len(NO_ANSWER_CASES)}')
if not CAN_SCORE:
    print(f'⚠️ Scores NON calculés : annoter au moins {MIN_ANNOTATED} questions, puis renseigner ANNOTATIONS_CSV et relancer.')

---
## Étape 8 — Mesurer

Pour une question, `G` est l'ensemble des documents pertinents et `R` la liste des `K` documents distincts retournés :

- **Precision@k** = |R ∩ G| / k — part des résultats qui sont pertinents ;
- **Recall@k** = |R ∩ G| / |G| — part des documents pertinents retrouvés ;
- **Hit@k** = 1 si au moins un document pertinent figure dans R ;
- **RR** = 1 / rang du premier document pertinent (0 s'il est absent) ; **MRR** en est la moyenne.

Quand une question n'a qu'**un** document pertinent, Precision@5 ne peut pas dépasser 0,2 : regardez surtout Recall@k et MRR.

In [ ]:
def retrieval_metrics(ranked_docs, gold, k=K):
    assert len(ranked_docs) == len(set(ranked_docs)), 'Dédupliquer les documents avant de mesurer.'
    gold = set(gold)
    if not gold: raise ValueError('Question sans référence : évaluer l’abstention séparément.')
    top = ranked_docs[:k]
    found = [d for d in top if d in gold]
    rr = next((1.0/r for r, d in enumerate(top, 1) if d in gold), 0.0)
    return {'precision':len(found)/k, 'recall':len(found)/len(gold), 'hit':float(bool(found)), 'rr':rr}

assert retrieval_metrics(['a','b','c'], {'a','c'}, 3) == {'precision':2/3, 'recall':1.0, 'hit':1.0, 'rr':1.0}
assert retrieval_metrics(['x','a'], {'a'}, 1) == {'precision':0.0, 'recall':0.0, 'hit':0.0, 'rr':0.0}
assert retrieval_metrics(['x','a'], {'a','b'}, 2) == {'precision':0.5, 'recall':0.5, 'hit':1.0, 'rr':0.5}
print('Tests des métriques : OK')

rows = []
if CAN_SCORE:
    for case in EVAL_CASES:
        for name, fn in STRATEGIES.items():
            t0 = time.perf_counter()
            ranked = to_documents(fn(case['question']), K)
            latency = (time.perf_counter()-t0)*1000
            rows.append({'strategie':name, 'question_id':case['question_id'], 'domain_cible':case['domain_cible'],
                         'type':case['type'], 'question':case['question'], 'attendus':';'.join(sorted(case['refs'])),
                         'retrouves':';'.join(ranked), 'latence_ms':latency, **retrieval_metrics(ranked, case['refs'], K)})
EVAL_DF = pd.DataFrame(rows)
if EVAL_DF.empty:
    print('Aucune mesure : le jeu de référence est insuffisant.')
else:
    SUMMARY = (EVAL_DF.groupby('strategie', sort=False)
               .agg(questions=('question_id','count'), precision_at_k=('precision','mean'), recall_at_k=('recall','mean'),
                    hit_at_k=('hit','mean'), mrr=('rr','mean'), latence_ms=('latence_ms','mean')).round(3))
    print(f'Évaluation au niveau DOCUMENT, k = {K}, {len(EVAL_CASES)} questions ({origin})')
    display(SUMMARY)
    SUMMARY[['recall_at_k','hit_at_k','mrr']].plot(kind='bar', figsize=(9,3.6), ylim=(0,1.05), rot=15)
    plt.title(f'Retrieval au niveau document (k = {K}) — {origin}'); plt.ylabel('Moyenne sur les questions')
    plt.tight_layout(); plt.show()
    if DATASET_KIND != 'live':
        print('⚠️ Mesures sur documents FICTIFS : elles illustrent la méthode, pas la qualité sur le site UPPA.')

### Lire le tableau avec prudence

Une moyenne cache des gains **et** des pertes. Le tableau suivant compare chaque stratégie à la référence dense, **question par question** : combien de questions s'améliorent, se dégradent ou ne changent pas (sur le rang du premier document pertinent).

Avec moins d'une trentaine de questions, un écart de quelques points peut venir d'une ou deux questions seulement : ne concluez pas à une supériorité générale.

In [ ]:
if not EVAL_DF.empty:
    pivot = EVAL_DF.pivot(index='question_id', columns='strategie', values='rr')
    baseline = 'A — Dense'
    duel = []
    for name in pivot.columns:
        if name == baseline: continue
        diff = pivot[name] - pivot[baseline]
        duel.append({'comparaison':f'{name}  vs  {baseline}', 'améliorées':int((diff > 1e-9).sum()),
                     'dégradées':int((diff < -1e-9).sum()), 'inchangées':int((diff.abs() <= 1e-9).sum()),
                     'écart moyen de RR':round(float(diff.mean()), 3)})
    display(pd.DataFrame(duel))
    by_domain = EVAL_DF.pivot_table(index='domain_cible', columns='strategie', values='recall', aggfunc='mean', sort=False).round(3)
    print('Recall@k par domaine cible :')
    display(by_domain)

---
## Étape 9 — Diagnostiquer les erreurs

Les chiffres disent **combien** ; il faut encore comprendre **pourquoi**. Pour chaque question où la référence dense échoue (aucun document pertinent dans les `K` premiers), regardez ce que les autres stratégies retrouvent, puis ouvrez les passages.

Causes fréquentes : page pertinente absente du corpus, chunk tronqué, sigle ou code que le modèle dense ne distingue pas, page « hub » qui capte la requête, formulation de la question très éloignée du texte.

In [ ]:
if not EVAL_DF.empty:
    hits = EVAL_DF.pivot(index='question_id', columns='strategie', values='hit')
    failures = hits.index[hits['A — Dense'] == 0]
    print(f'Questions où la recherche dense échoue : {len(failures)} / {len(hits)}')
    if len(failures):
        detail = EVAL_DF[EVAL_DF.question_id.isin(failures)][['question_id','question','strategie','hit','rr','retrouves','attendus']]
        display(detail.sort_values(['question_id','strategie']))
    never = hits.index[hits.max(axis=1) == 0]
    if len(never):
        print('Aucune stratégie ne retrouve la source pour :', list(never))
        print('→ Vérifier que la page est dans le corpus et que son contenu utile n’a pas été perdu au découpage.')

### Questions sans réponse dans le corpus

Un bon assistant doit savoir dire « je ne trouve pas ». Le score du meilleur passage est-il plus faible quand le corpus ne contient pas la réponse ? La cellule compare le **cosinus du premier résultat** pour les questions avec et sans réponse.

Ce n'est **pas** un détecteur fiable : les deux distributions se recouvrent souvent, et un seuil réglé sur vos questions ne vaut pas pour d'autres. L'abstention se juge surtout sur la réponse générée (étape 10).

In [ ]:
if NO_ANSWER_CASES and EVAL_CASES:
    top_score = lambda q: float(np.max(dense_scores(q)))
    with_answer = [top_score(c['question']) for c in EVAL_CASES]
    without = [top_score(c['question']) for c in NO_ANSWER_CASES]
    print(f'Cosinus du 1er résultat — avec réponse : médiane {np.median(with_answer):.3f} (n={len(with_answer)}) '
          f'| sans réponse : médiane {np.median(without):.3f} (n={len(without)})')
    print('Recouvrement : la plus haute valeur « sans réponse » est', round(max(without), 3),
          '; la plus basse « avec réponse » est', round(min(with_answer), 3))
else:
    print('Ajouter des questions de type « sans_reponse » dans la grille pour étudier l’abstention.')

---
## Étape 10 — La génération se juge à part

Un retrieval excellent ne garantit pas une réponse fidèle. Le Notebook 03 exporte une grille (`grille_evaluation_*.csv`) à remplir **à la main** après lecture des sources, avec `oui` ou `non` dans :

- `answer_faithful_manual` — chaque affirmation est-elle soutenue par le passage cité ?
- `answer_correct_manual` — la réponse est-elle exacte et complète ?
- `admission_year_valid_manual` — l'année ou la campagne citée est-elle établie par la source ?

Si vous fournissez l'archive du Notebook 03 avec cette grille complétée, la cellule en fait la synthèse. Tant que les cases valent `A_EVALUER`, **aucun taux n'est calculé**.

In [ ]:
GENERATION_SUMMARY = None
if NB03_ZIP:
    grid = pd.read_csv(io.BytesIO(read_member(NB03_ZIP, r'(?:^|/)grille_evaluation_(?:uppa|synthetic_demo)\.csv$')), dtype=str).fillna('')
    verdict = lambda v: {'oui':1.0, 'yes':1.0, '1':1.0, 'non':0.0, 'no':0.0, '0':0.0}.get(str(v).strip().casefold())
    lines = []
    for column in ('answer_faithful_manual','answer_correct_manual','admission_year_valid_manual'):
        if column not in grid.columns: continue
        values = [verdict(v) for v in grid[column]]
        judged = [v for v in values if v is not None]
        lines.append({'critère':column, 'réponses jugées':len(judged), 'réponses à évaluer':len(values)-len(judged),
                      'taux de oui':round(float(np.mean(judged)), 3) if len(judged) >= 3 else 'trop peu de jugements'})
    GENERATION_SUMMARY = pd.DataFrame(lines)
    display(GENERATION_SUMMARY)
    if 'citations_unknown' in grid.columns:
        print('Réponses citant une balise inexistante :', int((grid['citations_unknown'].str.strip() != '').sum()), '/', len(grid))
else:
    print('Archive du Notebook 03 non fournie : la fidélité des réponses reste à évaluer manuellement.')

---
## Étape 11 — Exporter les résultats

L'archive contient les mesures par question, le tableau de synthèse, la grille de 45 questions, le catalogue des documents et un manifeste décrivant la configuration. Elle reste étiquetée `uppa` ou `synthetic_demo`.

In [ ]:
manifest = {
  'schema_version':'uppa_rag_evaluation_v1',
  'created_at':datetime.now(timezone.utc).isoformat(timespec='seconds'),
  'dataset_kind':DATASET_KIND, 'input_manifest':MANIFEST, 'embedding_model':EMBEDDING_MODEL,
  'rerank_model':RERANK_MODEL if reranker is not None else None,
  'strategies':list(STRATEGIES), 'k_documents':K, 'n_candidate_chunks':N_CANDIDATES, 'rrf_k':RRF_K,
  'evaluation_level':'document', 'n_questions_with_references':len(EVAL_CASES), 'n_questions_without_answer':len(NO_ANSWER_CASES),
  'scores_computed':not EVAL_DF.empty, 'annotation_origin':origin,
  'note':'Résultats fictifs non officiels' if DATASET_KIND != 'live' else 'Mesures sur annotations manuelles des étudiants',
}
(EXPORT_DIR/'evaluation_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
if not EVAL_DF.empty:
    EVAL_DF.to_csv(EXPORT_DIR/f'evaluation_retrieval_{SUFFIX}.csv', index=False, encoding='utf-8-sig')
    SUMMARY.to_csv(EXPORT_DIR/f'synthese_retrieval_{SUFFIX}.csv', encoding='utf-8-sig')
if GENERATION_SUMMARY is not None:
    GENERATION_SUMMARY.to_csv(EXPORT_DIR/f'synthese_generation_{SUFFIX}.csv', index=False, encoding='utf-8-sig')
archive = BASE/f'UPPA_Notebook04_evaluation_{SUFFIX}.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(EXPORT_DIR.glob('*')):
        if p.is_file(): z.write(p, Path('exports')/p.name)
print('Archive :', archive)
print('Dans Colab : from google.colab import files; files.download(str(archive))')

---
## Étape 12 — Travail à rendre

1. La grille de **45 questions annotées** (12 par domaine, 9 transversales), avec au moins 5 questions `sans_reponse`.
2. Le **tableau comparatif** A / B / C / D sur ces questions, pour l'index V2 **et** l'index V3 (relancer le notebook avec chaque archive).
3. Le tableau « améliorées / dégradées / inchangées » et son interprétation : la stratégie retenue fait-elle aussi des perdants ?
4. **Trois analyses d'erreurs** détaillées : question, document attendu, ce qui a été retrouvé, cause probable, correction proposée.
5. Une expérience à un seul facteur : faire varier `K`, `N_CANDIDATES` **ou** `RRF_K`, et commenter l'effet.
6. La synthèse de la **grille de fidélité** du Notebook 03 (au moins 10 réponses jugées).
7. La configuration finale justifiée, avec ses limites (taille de l'échantillon, couverture du corpus, fraîcheur des pages d'admission).

**Approfondissement Industry 4.0 :** questions portant sur des codes d'UE, capteurs, automatismes — où BM25 aide-t-il ?
**Approfondissement Logistique et services :** questions reformulées sans les mots de la fiche — où la recherche dense reste-t-elle indispensable ?

### Ressources
- [Robertson & Zaragoza — The Probabilistic Relevance Framework: BM25 and Beyond](https://www.staff.city.ac.uk/~sbrp622/papers/foundations_bm25_review.pdf)
- [Cormack, Clarke & Büttcher — Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf)
- [Sentence Transformers — Cross-Encoders](https://sbert.net/docs/cross_encoder/usage/usage.html)
- [rank-bm25](https://github.com/dorianbrown/rank_bm25)

*Ce notebook est un prototype pédagogique, indépendant de l'UPPA. Aucun résultat obtenu sur le corpus fictif ne doit être présenté comme une mesure sur les pages officielles.*